<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 6 · Visualización de datos

La semana pasada construiste la tabla mensual de indicadores y la cuadraste contra una cifra de
control. Es correcta, es reproducible y **nadie la va a leer**: son 154 filas de números. Hoy la
conviertes en cinco figuras que se entienden en tres segundos, y aprendes las reglas para que esas
figuras no digan más de lo que los datos aguantan. Porque un gráfico no es el adorno del análisis: es
el análisis, y es también la forma más rápida de mentir sin darse cuenta.

> **Hoy haces** · El repertorio mínimo con matplotlib y seaborn: línea, barras ordenadas, dispersión,
> histograma y caja, cada uno con la pregunta que contesta (90 min). Reproduces el cuarteto de Anscombe
> y compruebas que cuatro conjuntos con los mismos estadísticos tienen cuatro formas distintas. Repasas
> las distorsiones visuales más frecuentes y mides cuánto exagera un eje truncado. Reescribes cada
> título para que lleve la conclusión.
>
> **Entrega** · Este cuaderno ejecutado, las cinco figuras del repertorio con título-conclusión y
> fuente al pie, el cuarteto de Anscombe con su tabla de estadísticos, y la pareja de barras truncada
> y desde cero con el factor de exageración calculado.
> Nombre de archivo: `lab_06_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              Path("/content/CursoAnalisisDatos_IA_2026/sitio/datos")]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    raise FileNotFoundError(
        "No encuentro la carpeta de datos. En Colab ejecuta primero:\n"
        "  !git clone https://github.com/<usuario>/CursoAnalisisDatos_IA_2026.git")

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Explorar, verificar, comunicar

Un gráfico se hace por tres razones distintas y solo una llega al informe.

**Explorar** es mirar rápido y feo, veinte figuras en cinco minutos, para saber qué hay.
**Verificar** es comprobar un supuesto: ¿la distribución es la que creo?, ¿hay un salto en marzo?
**Comunicar** es una sola figura, pensada, anotada y con la conclusión escrita encima.

El error habitual es enseñar los gráficos de explorar. Empecemos por el material: la tabla de la
semana pasada, reconstruida en una celda.

In [ ]:
ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
clientes = pd.read_csv(DATOS / "clientes.csv")
productos = pd.read_csv(DATOS / "productos.csv")
sucursales = pd.read_csv(DATOS / "sucursales.csv")
marketing = pd.read_csv(DATOS / "marketing_mensual.csv", parse_dates=["mes"])

clientes["ciudad"] = clientes["ciudad"].str.strip().str.title().replace({"Guayaquíl": "Guayaquil"})

v = (ventas
     .assign(monto=lambda d: d["cantidad"] * d["precio_unitario"] * (1 - d["descuento"]),
             mes=lambda d: d["fecha"].dt.to_period("M").dt.to_timestamp())
     .merge(clientes[["cliente_id", "ciudad", "tipo_cliente"]], on="cliente_id",
            how="left", validate="m:1")
     .merge(productos[["producto_id", "categoria", "subcategoria", "costo_unitario"]],
            on="producto_id", how="left", validate="m:1")
     .merge(sucursales[["sucursal_id", "canal"]], on="sucursal_id", how="left", validate="m:1"))
v["margen"] = v["monto"] - v["cantidad"] * v["costo_unitario"]

# Julio de 2026 solo tiene notas de crédito de junio: fuera de la serie, y dicho en voz alta.
serie = v.groupby("mes")["monto"].sum()
serie = serie[serie.index < "2026-07-01"]

print(f"{len(v):,} líneas · {len(serie)} meses completos · {v['monto'].sum():,.2f} de facturación neta")
print(f"(se excluye julio de 2026, que solo contiene devoluciones por {v.loc[v['mes'] >= '2026-07-01', 'monto'].sum():,.2f})\n")
print("La tabla que nadie lee, en formato ancho y aun así ilegible:")
print(serie.to_frame("facturación").T.round(0).to_string())

Treinta números en una línea. Están todos, son correctos y no dicen nada. Ahora los mismos treinta
números, dibujados.

## 2. Línea: el gráfico del tiempo

La línea es para una variable continua a lo largo del tiempo. Nada más. Si el eje horizontal no es
tiempo (o algo con orden natural), la línea sugiere una continuidad que no existe y hay que usar
barras.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(serie.index, serie.values, marker="o", markersize=4, linewidth=2, color="#4C72B0")

pico = serie.idxmax()
ax.annotate(f"dic-2025: {serie.max():,.0f}",
            xy=(pico, serie.max()), xytext=(-95, -18), textcoords="offset points",
            arrowprops=dict(arrowstyle="->", color="#C44E52"), color="#C44E52", fontweight="bold")
for anio in [2024, 2025]:
    d = pd.Timestamp(f"{anio}-12-01")
    ax.axvline(d, color="#C44E52", alpha=0.25, linestyle="--")

ax.set_title("Diciembre es el pico de los dos años completos: el negocio tiene estacionalidad, no tendencia")
ax.set_ylabel("facturación neta del mes")
ax.set_xlabel("")
ax.set_ylim(0, serie.max() * 1.15)
ax.text(0.99, -0.16, "Fuente: ventas_limpias.csv · Comercial Andina", transform=ax.transAxes,
        ha="right", fontsize=8, color="gray")
plt.tight_layout()
plt.show()

El eje empieza en cero, la línea tiene marcadores porque los puntos son mediciones reales y no una
función continua, y el título dice la conclusión en lugar de repetir «facturación mensual».

Que diciembre sea el máximo de los dos años completos es sugerente, pero un pico repetido dos veces
todavía no es estacionalidad. Para afirmarlo hay que plegar la serie sobre el calendario.

In [ ]:
perfil_mes = serie.groupby(serie.index.month).mean()
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
indice = perfil_mes / perfil_mes.mean() * 100

fig, ax = plt.subplots(figsize=(11, 3.8))
colores = ["#C44E52" if i in (12, 11, 5) else "#B0B0B0" for i in perfil_mes.index]
ax.bar([MESES[i - 1] for i in perfil_mes.index], perfil_mes.values, color=colores)
ax.axhline(perfil_mes.mean(), color="#4C72B0", linestyle="--", linewidth=1.2)
ax.text(0.2, perfil_mes.mean() * 1.02, f"media mensual {perfil_mes.mean():,.0f}",
        color="#4C72B0", fontsize=9)
for x, (mes, valor) in enumerate(indice.items()):
    ax.text(x, perfil_mes.iloc[x] + 1500, f"{valor:.0f}", ha="center", fontsize=8)

ax.set_title("Diciembre vende 78 % más que febrero: la campaña se planifica en octubre, no en diciembre")
ax.set_ylabel("facturación media del mes")
ax.text(0.99, -0.14, "Fuente: ventas_limpias.csv · índice 100 = mes medio", transform=ax.transAxes,
        ha="right", fontsize=8, color="gray")
plt.tight_layout()
plt.show()

print(f"diciembre {perfil_mes[12]:,.2f}  ·  febrero {perfil_mes[2]:,.2f}  "
      f"·  razón {perfil_mes[12] / perfil_mes[2]:.2f}")

📌 Diciembre está en el índice 136 y febrero en 76: **el mejor mes factura 1,78 veces el peor**. Con
eso ya hay una decisión: si el pico es diciembre y el inventario tarda seis semanas en llegar, la orden
de compra se firma en octubre. Ese es el salto de «gráfico bonito» a «gráfico que cambia algo».

## 3. Barras ordenadas: comparar categorías

Las barras comparan magnitudes entre categorías. Dos reglas y una prohibición:

1. **Ordenadas por valor**, no alfabéticamente. El orden es información gratis.
2. **Horizontales** si las etiquetas son largas.
3. **El eje empieza en cero. Siempre.** La longitud de la barra *es* el dato.

In [ ]:
por_ciudad = v.groupby("ciudad")["monto"].sum().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
alfabetico = por_ciudad.sort_index(ascending=False)
axes[0].barh(alfabetico.index, alfabetico.values, color="#B0B0B0")
axes[0].set_title("Orden alfabético: hay que leerlo dos veces", fontsize=11)

axes[1].barh(por_ciudad.index, por_ciudad.values,
             color=["#B0B0B0"] * 4 + ["#4C72B0"])
for y, valor in enumerate(por_ciudad.values):
    axes[1].text(valor + 15000, y, f"{valor / 1000:,.0f}k", va="center", fontsize=9)
axes[1].set_title("Ordenado: Quito factura 4,7 veces lo de Loja", fontsize=11)
axes[1].set_xlim(0, por_ciudad.max() * 1.18)

for a in axes:
    a.set_xlabel("facturación neta acumulada 2024-2026")
fig.suptitle("El orden no es estética: es la mitad del mensaje", fontsize=12, y=1.04)
plt.tight_layout()
plt.show()

print(f"Quito {por_ciudad['Quito']:,.2f} · Loja {por_ciudad['Loja']:,.2f} "
      f"· razón {por_ciudad['Quito'] / por_ciudad['Loja']:.2f}")
print(f"Quito y Guayaquil concentran el "
      f"{por_ciudad[['Quito', 'Guayaquil']].sum() / por_ciudad.sum():.1%} de la facturación.")

Mismos datos, mismo tipo de gráfico, mismos colores. El de la derecha se lee en dos segundos y el de
la izquierda no. La única diferencia es un `sort_values()`.

## 4. Dispersión: relación entre dos variables

La dispersión es el gráfico de «¿esto tiene que ver con aquello?». Es también el que más se
sobreinterpreta, porque el ojo humano encuentra rectas donde solo hay nubes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
canales = [("inversion_radio", "Radio"), ("inversion_digital", "Digital"), ("inversion_volantes", "Volantes")]

for ax, (col, nombre) in zip(axes, canales):
    r = marketing[col].corr(marketing["ventas_mes"])
    ax.scatter(marketing[col], marketing["ventas_mes"], s=45, alpha=0.75, color="#4C72B0")
    ax.set_xlabel(f"inversión mensual en {nombre.lower()}")
    ax.set_title(f"{nombre}   r = {r:+.2f}", fontsize=11,
                 color="#C44E52" if abs(r) > 0.5 else "gray")

axes[0].set_ylabel("ventas del mes")
fig.suptitle("De los tres canales, solo volantes se mueve con las ventas (r = 0,94)", fontsize=12, y=1.02)
axes[0].text(0.0, -0.28, "Fuente: marketing_mensual.csv · 31 meses", transform=axes[0].transAxes,
             fontsize=8, color="gray")
plt.tight_layout()
plt.show()

print(marketing[["inversion_radio", "inversion_digital", "inversion_volantes", "ventas_mes"]]
      .corr()["ventas_mes"].round(3).to_string())

⚠️ La lectura tentadora es «los volantes funcionan y hay que invertir más». El gráfico **no dice eso**.
Dice que los dos números suben juntos, r = 0,94, y eso es compatible con tres historias distintas: los
volantes empujan las ventas, o el presupuesto de volantes se fija cada mes como porcentaje de la venta
esperada, o las dos cosas responden a la temporada. Con estos datos no se puede separar. La respuesta
está en la semana 9, con diseño experimental, y el número se convierte en modelo en la semana 12.

Por ahora, la frase honesta al pie de esa figura es: *«la inversión en volantes y las ventas se mueven
juntas; con estos datos no se puede afirmar cuál mueve a cuál»*.

## 5. Histograma y caja: la forma de una distribución

El histograma responde «¿cómo se reparten los valores?» y es la única forma de ver que un promedio no
describe a nadie. La caja responde lo mismo de forma comprimida y sirve para **comparar** varios
grupos a la vez.

In [ ]:
ticket = v.query("es_devolucion == False").groupby("factura_id")["monto"].sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(ticket, bins=60, color="#4C72B0", edgecolor="white")
axes[0].axvline(ticket.mean(), color="#C44E52", linewidth=2,
                label=f"media {ticket.mean():,.2f}")
axes[0].axvline(ticket.median(), color="#55A868", linewidth=2,
                label=f"mediana {ticket.median():,.2f}")
axes[0].set_title("Dos jorobas, no una campana", fontsize=11)
axes[0].set_xlabel("ticket de la factura")
axes[0].set_ylabel("número de facturas")
axes[0].legend()

tipo_factura = (v.query("es_devolucion == False")
                .groupby(["tipo_cliente", "factura_id"])["monto"].sum().reset_index())
sns.boxplot(data=tipo_factura, x="tipo_cliente", y="monto", ax=axes[1],
            hue="tipo_cliente", palette=["#C44E52", "#4C72B0"], legend=False)
axes[1].set_title("Separadas, cada una sí tiene un centro", fontsize=11)
axes[1].set_xlabel("")
axes[1].set_ylabel("ticket de la factura")

fig.suptitle("El ticket medio de 180,49 no le corresponde a ningún cliente de Comercial Andina",
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

print(tipo_factura.groupby("tipo_cliente")["monto"]
      .agg(facturas="size", media="mean", mediana="median", p25=lambda s: s.quantile(.25),
           p75=lambda s: s.quantile(.75)).round(2).to_string())

📌 La media global es 180,49 y la mediana 30,22: el histograma explica por qué. Hay 10 385 facturas
minoristas con una mediana de 20,56 y 5 542 mayoristas con una mediana de 459,53. **Ningún cliente
factura 180.** La caja de la derecha es la misma información en un cuarto del espacio, y por eso es la
que va al informe cuando hay que comparar grupos.

Las dos figuras contestan preguntas distintas: el histograma dice *qué forma tiene*, la caja dice
*cuánto se diferencian los grupos*. Elegir el gráfico es elegir la pregunta.

## 6. El cuarteto de Anscombe

En 1973 el estadístico Francis Anscombe construyó cuatro conjuntos de once puntos con **la misma media
en x, la misma media en y, la misma varianza, la misma correlación y la misma recta de regresión**.
Los construyes tú aquí, en código, y compruebas los estadísticos antes de dibujar nada.

In [ ]:
x_comun = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], dtype=float)
anscombe = {
    "I":   (x_comun, np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])),
    "II":  (x_comun, np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])),
    "III": (x_comun, np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])),
    "IV":  (np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], dtype=float),
            np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])),
}

filas = []
for nombre, (x, y) in anscombe.items():
    pendiente, corte = np.polyfit(x, y, 1)
    filas.append(dict(conjunto=nombre, n=len(x), media_x=x.mean(), var_x=x.var(ddof=1),
                      media_y=y.mean(), var_y=y.var(ddof=1),
                      correlacion=np.corrcoef(x, y)[0, 1],
                      recta=f"y = {corte:.2f} + {pendiente:.2f}x"))
estadisticos = pd.DataFrame(filas).set_index("conjunto")
print(estadisticos.round(2).to_string())

redondeados = estadisticos.drop(columns="recta").round(
    {"n": 0, "media_x": 2, "var_x": 2, "media_y": 2, "var_y": 1, "correlacion": 2})
print(f"\n¿los cuatro conjuntos dan los mismos estadísticos? "
      f"{bool(redondeados.nunique().max() == 1)}")
print(f"¿y la misma recta? {estadisticos['recta'].nunique() == 1} → {estadisticos['recta'].iloc[0]}")

Media de x igual a 9,00, varianza de x igual a 11,00, media de y igual a 7,50, varianza de y igual a
4,1 y correlación de 0,82 en los cuatro conjuntos, más la misma recta ajustada, `y = 3,00 + 0,50x`. Si tu informe dice «la correlación es 0,82 y la recta es esta», los cuatro conjuntos producen
exactamente el mismo informe. Ahora dibújalos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
lecturas = {"I": "relación lineal razonable",
            "II": "es una curva: la recta es el modelo equivocado",
            "III": "un solo punto atípico gira la recta",
            "IV": "un solo punto crea toda la correlación"}

for ax, (nombre, (x, y)) in zip(axes.ravel(), anscombe.items()):
    pendiente, corte = np.polyfit(x, y, 1)
    rejilla = np.array([2, 20])
    ax.plot(rejilla, corte + pendiente * rejilla, color="#C44E52", linewidth=1.5)
    ax.scatter(x, y, s=55, color="#4C72B0", zorder=3, edgecolor="white")
    ax.set_title(f"{nombre} · {lecturas[nombre]}", fontsize=10)
    ax.set_xlim(2, 20)
    ax.set_ylim(2, 14)

fig.suptitle("Mismos estadísticos, cuatro historias: r = 0,82 no describe una relación",
             fontsize=12, y=0.99)
plt.tight_layout()
plt.show()

📌 Este es el argumento entero de la semana en una figura. **Los estadísticos resumen; los gráficos
describen.** El conjunto II tiene una curva perfecta y la recta lo destroza. El III tiene diez puntos
alineados y un atípico que gira la recta. El IV no tiene ninguna relación: hay once puntos, diez en la
misma vertical, y la correlación de 0,82 la produce un único punto en x = 19.

La consecuencia práctica: **antes de reportar una correlación, dibújala**. Cuesta una línea de código y
es la diferencia entre un hallazgo y un accidente.

## 7. Honestidad visual: el catálogo de distorsiones

Casi nadie miente a propósito con un gráfico. Se miente con las opciones por defecto y con la
costumbre. Estas son las cuatro que más aparecen, y las cuatro se corrigen sin perder información.

In [ ]:
sub = v.groupby("subcategoria")["monto"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].pie(sub.values, labels=sub.index, autopct="%1.0f%%", textprops={"fontsize": 7.5})
axes[0].set_title("Pastel con 12 porciones: ¿cuál es la tercera?", fontsize=11)

axes[1].barh(sub.index[::-1], sub.values[::-1], color="#B0B0B0")
axes[1].barh(sub.index[:3][::-1], sub.values[:3][::-1], color="#4C72B0")
axes[1].set_title(f"Las mismas 12, en barras: la tercera es {sub.index[2]}", fontsize=11)
axes[1].set_xlabel("facturación neta acumulada")

fig.suptitle("El ojo compara longitudes bien y ángulos mal: el pastel solo sirve con 2 o 3 categorías",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(sub.head(4).round(2).to_string())
print(f"\nlas tres primeras subcategorías: {sub.head(3).sum() / sub.sum():.1%} de la facturación")

**El pastel con doce porciones.** El ojo humano compara longitudes con precisión y ángulos con
un error del orden del 25 %. Con dos o tres categorías el pastel funciona; con doce, no se puede ni
ordenar el ranking. La regla operativa: si tienes que poner los porcentajes para que se entienda, el
gráfico no está haciendo su trabajo.

**El doble eje.** Dos escalas independientes en la misma figura permiten poner las dos curvas donde
uno quiera y fabricar la relación que uno quiera.

In [ ]:
mkt = marketing[marketing["mes"] < "2026-07-01"].copy()

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4))
ax1 = axes[0]
ax1.plot(mkt["mes"], mkt["ventas_mes"], color="#4C72B0", linewidth=2, label="ventas")
ax2 = ax1.twinx()
ax2.plot(mkt["mes"], mkt["inversion_radio"], color="#C44E52", linewidth=2, label="radio")
ax1.set_title("Doble eje: «la radio arrastra las ventas»", fontsize=11)
ax1.grid(False); ax2.grid(False)

base = axes[1]
base.plot(mkt["mes"], mkt["ventas_mes"] / mkt["ventas_mes"].iloc[0] * 100,
          color="#4C72B0", linewidth=2, label="ventas")
base.plot(mkt["mes"], mkt["inversion_radio"] / mkt["inversion_radio"].iloc[0] * 100,
          color="#C44E52", linewidth=2, label="inversión en radio")
base.axhline(100, color="gray", linewidth=0.8)
base.set_title("Misma escala (base 100): no se parecen en nada", fontsize=11)
base.legend(fontsize=9)

fig.suptitle(f"La correlación real entre radio y ventas es {mkt['inversion_radio'].corr(mkt['ventas_mes']):+.2f}",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

Izquierda y derecha son **los mismos dos números**. En la izquierda parece que la radio y las ventas
comparten un patrón; en la derecha, con una sola escala en base 100, se ve que la correlación real es
−0,10. El doble eje no está prohibido, pero exige justificar por qué las dos escalas están donde están;
casi nunca hay respuesta.

**El 3D decorativo** y **las etiquetas giradas 45 grados** completan la lista. Ninguno añade
información y los dos añaden error de lectura. La regla común: si un elemento del gráfico no codifica
un dato, sobra.

## 8. El título que concluye

El título por defecto —«Facturación por ciudad»— repite lo que ya dicen los ejes y desperdicia el
único renglón que todo el mundo lee. El título útil trae la conclusión y, si se puede, la decisión.

In [ ]:
margen_ciudad = (v.groupby("ciudad")
                 .agg(facturacion=("monto", "sum"), margen=("margen", "sum"))
                 .assign(margen_pct=lambda d: d["margen"] / d["facturacion"] * 100))
top = margen_ciudad.sort_values("facturacion", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8), sharey=True)
for ax, titulo in zip(axes, ["Facturación y margen por ciudad",
                             "Quito aporta el 36 % de la facturación y rinde el mismo margen que el resto"]):
    ax.bar(top.index, top["facturacion"], color="#B0B0B0")
    ax.set_title(titulo, fontsize=11)
    ax.set_ylabel("facturación neta")
axes[1].bar("Quito", top.loc["Quito", "facturacion"], color="#4C72B0")
for x, (ciudad, fila) in enumerate(top.iterrows()):
    axes[1].text(x, fila["facturacion"] + 20000, f"{fila['margen_pct']:.1f} % margen",
                 ha="center", fontsize=8.5)
axes[1].set_ylim(0, top["facturacion"].max() * 1.2)
plt.tight_layout()
plt.show()

print(f"Quito: {top.loc['Quito', 'facturacion'] / top['facturacion'].sum():.1%} de la facturación, "
      f"{top.loc['Quito', 'margen_pct']:.2f} % de margen contra "
      f"{margen_ciudad['margen_pct'].mean():.2f} % de media.")

El título de la izquierda describe el gráfico. El de la derecha describe el negocio, y por eso la
figura de la derecha se puede leer sin que nadie la explique. La prueba: si tapas el título y alguien
del equipo no llega a la misma frase, el gráfico no comunica.

### 🌶️ Ejercicio 1 — Guiado

Rehaz la figura de la sección 3 —barras por ciudad— pero abriendo por tipo de cliente: dos barras por
ciudad, mayorista y minorista. Ordena por facturación total, pon el eje en cero, escribe el
título-conclusión y añade la fuente al pie. Después contesta en una celda de texto: ¿qué ciudad
depende más del mayorista, y qué haría el gerente comercial con esa información?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: base = v.pivot_table(index="ciudad", columns="tipo_cliente", values="monto", aggfunc="sum")
# Pista 2: base.plot(kind="barh", stacked=True) hace el apilado en una línea
# Pista 3: el porcentaje se calcula con base.div(base.sum(axis=1), axis=0)
# El título tiene que contener un número y un verbo

### 🔥 Desafío

Construye tu propio cuarteto: **cuatro subconjuntos de las ventas de Comercial Andina que tengan
ticket medio parecido y formas de distribución distintas**. Sirven ciudades, canales, categorías o
meses. Demuestra con una tabla que las medias se parecen y con cuatro histogramas que las
distribuciones no. Es el argumento de Anscombe aplicado a datos de negocio, y es el que se usa para
defender por qué no basta con reportar promedios.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: candidatos naturales: canal (Online / Tienda), tipo_cliente, categoria, ciudad
# Pista 2: compara media, mediana, desviación estándar y asimetría con .skew()
# Pista 3: usa el MISMO rango de ejes en los cuatro histogramas, o la comparación no vale nada

### 🎯 Reto en clase (15 min)

Clínica de rediseño, en equipos. Cada equipo toma **una** de las figuras de este cuaderno y la rehace
para empeorarla todo lo posible sin borrar ni un dato: eje truncado, orden alfabético, doce colores,
leyenda redundante, título neutro, 3D. Se la pasa al equipo de al lado, que tiene diez minutos para
enumerar los cambios y estimar **en qué porcentaje se distorsiona la lectura**. Gana el equipo que
detecta más distorsiones con su cifra, no el que hizo el gráfico más feo.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: para medir la distorsión de un eje truncado, compara la razón entre alturas visuales
#   razon_real      = valor_max / valor_min
#   razon_percibida = (valor_max - base_eje) / (valor_min - base_eje)
# El factor de exageración es razon_percibida / razon_real

## La trampa de hoy

⚠️ **Empezar el eje vertical en un valor distinto de cero en un gráfico de barras.** En una barra el
dato es la longitud; si cortas la base, la longitud deja de ser el dato y pasa a ser la diferencia
contra un número que elegiste tú. El caso real de Comercial Andina: el margen por ciudad, que va del
44,04 % al 44,29 %. Un cuarto de punto porcentual.

In [ ]:
m = margen_ciudad["margen_pct"].sort_values(ascending=False)
BASE = 44.0

razon_real = m.max() / m.min()
razon_percibida = (m.max() - BASE) / (m.min() - BASE)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
axes[0].bar(m.index, m.values, color="#C44E52")
axes[0].set_ylim(BASE, m.max() + 0.03)
axes[0].set_title(f"Eje desde {BASE}: «Loja rinde {razon_percibida:.1f} veces más que Guayaquil»",
                  fontsize=11, color="#C44E52")
axes[0].set_ylabel("margen sobre ventas (%)")

axes[1].bar(m.index, m.values, color="#4C72B0")
axes[1].set_ylim(0, 55)
axes[1].set_title(f"Eje desde cero: las cinco ciudades rinden igual ({razon_real:.3f} veces)",
                  fontsize=11, color="#4C72B0")
axes[1].set_ylabel("margen sobre ventas (%)")
for x, valor in enumerate(m.values):
    axes[1].text(x, valor + 1, f"{valor:.2f} %", ha="center", fontsize=9)

fig.suptitle("Los mismos cinco números, dos conclusiones opuestas", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"margen máximo (Loja)      : {m.max():.2f} %")
print(f"margen mínimo (Guayaquil) : {m.min():.2f} %")
print(f"diferencia real           : {m.max() - m.min():.2f} puntos porcentuales "
      f"({razon_real:.4f} veces, es decir un {razon_real * 100 - 100:.2f} % más)")
print(f"diferencia percibida con el eje en {BASE} : {razon_percibida:.2f} veces")
print(f"FACTOR DE EXAGERACIÓN     : ×{razon_percibida / razon_real:.1f}")

📌 Loja rinde un **0,58 % más** que Guayaquil. Con el eje cortado en 44,0 la barra de Loja mide
**7,43 veces** la de Guayaquil: el gráfico exagera la diferencia real por un factor de 7,4. Nadie
mintió, nadie borró un dato y las dos figuras salen del mismo `DataFrame`. Lo único que cambió fue una
línea, `set_ylim`.

Las tres reglas que se derivan y que ya no se negocian:

1. **Barras: el eje empieza en cero.** Sin excepciones. Si la diferencia no se ve con el eje en cero,
   es que la diferencia es pequeña, y eso también es un hallazgo.
2. **Si de verdad hay que ampliar**, se cambia de gráfico: un gráfico de puntos, una línea o un
   gráfico de la *diferencia* contra la media pueden empezar donde haga falta, porque ahí el dato es
   la posición y no la longitud.
3. **Escribe la magnitud en el texto.** «Loja rinde 0,25 puntos porcentuales más» es una frase que no
   se puede distorsionar con un `set_ylim`.

## Entregable

Sube `lab_06_apellido.ipynb` con:

- Las cinco figuras del repertorio —línea, barras ordenadas, dispersión, histograma y caja— cada una
  con título-conclusión, eje en cero cuando corresponde y la fuente al pie.
- El cuarteto de Anscombe reconstruido en código, la tabla que demuestra que los cinco estadísticos
  coinciden y la figura que demuestra que las formas no.
- La pareja de barras del margen por ciudad, truncada y desde cero, con el factor de exageración
  calculado (×7,4) impreso por el código, no escrito a mano.
- Los tres ejercicios, con la frase de negocio que acompaña a cada figura.
- Una fila en la bitácora de prompts: le pediste al asistente que argumentara **en contra** de la
  conclusión de una de tus figuras usando esa misma figura. Si lo consiguió, anota qué cambiaste.

## Para tu equipo

- Los tres gráficos del caso se entregan esta semana con la conclusión escrita en el título y la
  fuente citada al pie. Un título que nombra el eje se devuelve sin corregir.
- Antes de elegir el tipo de gráfico, escriban la pregunta. Tiempo → línea. Comparar categorías →
  barras ordenadas. Relación → dispersión. Forma de una variable → histograma. Comparar formas →
  cajas. Si ninguna encaja, la pregunta todavía no está escrita.
- Pásenle sus tres figuras a otro equipo sin los títulos y pídanles que escriban qué concluyen. Si lo
  que escriben no es lo que ustedes querían decir, el problema es de la figura, no del lector.